In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [3]:
spark = SparkSession.builder.appName("SalesAnalytics").getOrCreate()

In [4]:
sales_data = [(1,"2025-01-01",101,1001,2,1200),
              (2,"2025-01-02",102,1002,1,800),
              (3,"2025-01-03",103,1003,4,2000),
              (4,"2025-01-04",103,1003,3,1500),
              (5,"2025-01-05",105,1005,2,1800),
              (6,"2025-01-06",103,1001,1,2200),
              (7,"2025-01-07",102,1003,None,3000) 
            ]
sales_df= spark.createDataFrame(sales_data, 
                                ["OrderID", "OrderDate", "CustomerID", "ProductID", "Quantity", "SalesAmount"])


In [5]:
customer_data = [(101,"Alice","North","2025-01-01"),
                 (102,"Bob","South","2025-01-02"),   
                 (103,"Charlie","East","2025-01-03"),
                 (104,"David","West","2025-01-04"),
                 (105,"Eve","North","2025-01-05")]
customer_df = spark.createDataFrame(customer_data, 
                                    ["CustomerID", "CustomerName", "Region", "JoinDate"])

In [6]:
product_data = [(1001,"Laptop","Electronics"),
                (1002,"Phone","Electronics"),
                (1003,"Table","Furniture"),
                (1004,"Chair","Furniture"),
                (1005,"Headphones","Electronics")]

product_df = spark.createDataFrame(product_data, 
                                    ["ProductID", "ProductName", "Category"])

In [7]:
sales_df = sales_df.withColumn("Quantity", coalesce(col("Quantity"), lit(1)))

In [9]:
sales_df = sales_df.withColumn("Revenue", col("Quantity") * col("SalesAmount") )

In [10]:
sales_df

DataFrame[OrderID: bigint, OrderDate: string, CustomerID: bigint, ProductID: bigint, Quantity: bigint, SalesAmount: bigint, Revenue: bigint]

In [12]:
sales_mart = sales_df.join(customer_df,"CustomerID","left") \
        .join(product_df, "ProductID", "left")

In [13]:
sales_mart.groupBy("Region").sum("Revenue").show()

+------+------------+
|Region|sum(Revenue)|
+------+------------+
| South|        3800|
|  East|       14700|
| North|        6000|
+------+------------+



In [14]:
spark

In [15]:
sales_mart

DataFrame[ProductID: bigint, CustomerID: bigint, OrderID: bigint, OrderDate: string, Quantity: bigint, SalesAmount: bigint, Revenue: bigint, CustomerName: string, Region: string, JoinDate: string, ProductName: string, Category: string]

In [17]:
sales_mart = sales_mart.withColumn("OrderDate",to_date("OrderDate"))

In [18]:
sales_mart

DataFrame[ProductID: bigint, CustomerID: bigint, OrderID: bigint, OrderDate: date, Quantity: bigint, SalesAmount: bigint, Revenue: bigint, CustomerName: string, Region: string, JoinDate: string, ProductName: string, Category: string]

In [20]:
sales_mart.groupby(month("OrderDate")).agg(sum("Revenue").alias("total_revenue")).show()

+----------------+-------------+
|month(OrderDate)|total_revenue|
+----------------+-------------+
|               1|        24500|
+----------------+-------------+



In [21]:
# Top 2 products in revenue 
sales_mart.groupBy("ProductName").agg(sum("Revenue").alias("total_revenue")).orderBy(desc("total_revenue")).show(2)

+-----------+-------------+
|ProductName|total_revenue|
+-----------+-------------+
|      Table|        15500|
|     Laptop|         4600|
+-----------+-------------+
only showing top 2 rows



In [22]:
# CLV  Customer Lifetime Value 
sales_mart.groupBy("CustomerID","CustomerName").agg(sum("Revenue").alias("total_revenue"), count("OrderID").alias("total_orders")).show() 

+----------+------------+-------------+------------+
|CustomerID|CustomerName|total_revenue|total_orders|
+----------+------------+-------------+------------+
|       105|         Eve|         3600|           1|
|       101|       Alice|         2400|           1|
|       102|         Bob|         3800|           2|
|       103|     Charlie|        14700|           3|
+----------+------------+-------------+------------+



In [24]:
sales_mart.select (       [  count(when (col(c).isNull(),c)).alias(c + "_null_count")             for c in  sales_mart.columns  ] ).show()

+--------------------+---------------------+------------------+--------------------+-------------------+----------------------+------------------+-----------------------+-----------------+-------------------+----------------------+-------------------+
|ProductID_null_count|CustomerID_null_count|OrderID_null_count|OrderDate_null_count|Quantity_null_count|SalesAmount_null_count|Revenue_null_count|CustomerName_null_count|Region_null_count|JoinDate_null_count|ProductName_null_count|Category_null_count|
+--------------------+---------------------+------------------+--------------------+-------------------+----------------------+------------------+-----------------------+-----------------+-------------------+----------------------+-------------------+
|                   0|                    0|                 0|                   0|                  0|                     0|                 0|                      0|                0|                  0|                     0|             

In [26]:
product_df.join(sales_df,"ProductId","left_anti").show()

+---------+-----------+---------+
|ProductID|ProductName| Category|
+---------+-----------+---------+
|     1004|      Chair|Furniture|
+---------+-----------+---------+



In [27]:
# Partitioning

In [28]:
# It is a Chunk of data    # Lives in memory  
# Core -> Task parallelly  

In [29]:
spark.sparkContext.defaultParallelism

8

In [30]:
df = spark.range(1, 5000000)

In [31]:
df.show(5)

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
+---+
only showing top 5 rows



In [32]:
sales_df = df.withColumn("CustomerID", floor(rand()*100000)) \
    .withColumn("RegionId",  floor(rand()*10)) \
    .withColumn("Amount", floor(rand()*1000))  

In [33]:
sales_df.show(5)

+---+----------+--------+------+
| id|CustomerID|RegionId|Amount|
+---+----------+--------+------+
|  1|     75131|       3|   875|
|  2|     98696|       1|   411|
|  3|     44973|       0|   421|
|  4|     47279|       5|   576|
|  5|     11216|       5|   310|
+---+----------+--------+------+
only showing top 5 rows



In [34]:
sales_df.count()

4999999

In [35]:
sales_df.rdd.getNumPartitions()

8

In [36]:
# Repartitioning 

In [39]:
df2 = sales_df.repartition(20)

In [40]:
df2.rdd.getNumPartitions()

20

In [41]:
# coalesce 

In [42]:
df3 = df2.coalesce(5)

In [43]:
df3.rdd.getNumPartitions()

5

In [44]:
# Column Based Partitioning (Hash Partitioning)

In [45]:
df_partition = sales_df.repartition("RegionId")

In [46]:
df_partition.rdd.getNumPartitions()

10

In [47]:
sales_df.write.mode("overwrite").partitionBy("RegionId").parquet("Files/Sales_Partitioned")

In [50]:
spark.read.parquet("Files/Sales_Partitioned").filter("RegionId = 5").count()  # Partition Pruning 

501480

In [51]:
df_read = spark.read.parquet("Files/Sales_Partitioned")

In [52]:
df_read.explain(True) 

== Parsed Logical Plan ==
Relation [id#716L,CustomerID#717L,Amount#718L,RegionId#719] parquet

== Analyzed Logical Plan ==
id: bigint, CustomerID: bigint, Amount: bigint, RegionId: int
Relation [id#716L,CustomerID#717L,Amount#718L,RegionId#719] parquet

== Optimized Logical Plan ==
Relation [id#716L,CustomerID#717L,Amount#718L,RegionId#719] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id#716L,CustomerID#717L,Amount#718L,RegionId#719] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/D:/DE 202512/CodeFiles/Files/Sales_Partitioned], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<id:bigint,CustomerID:bigint,Amount:bigint>



In [54]:
customer_df = sales_df.select("CustomerId").distinct() 

In [55]:
customer_df

DataFrame[CustomerId: bigint]

In [57]:
sales_df.join(customer_df,"CustomerId")  # data shuffling 

DataFrame[CustomerID: bigint, id: bigint, RegionId: bigint, Amount: bigint]

In [58]:
sales_df = sales_df.repartition("CustomerId")

In [59]:
customer_df = customer_df.repartition("CustomerId")

In [61]:
joined = sales_df.join(customer_df,"CustomerId")  # Less Shuffling and Faster Joins 

In [63]:
# DataSkewing   # One Partition has Huge data 

In [64]:
skew_df = sales_df.withColumn("RegionId", when(rand() < 0.8,1).otherwise(floor(rand()*10))     )

In [66]:
skew_df.groupBy("RegionId").count().show()

+--------+-------+
|RegionId|  count|
+--------+-------+
|       0|  99853|
|       7| 100459|
|       6| 100271|
|       9| 100393|
|       5|  99928|
|       1|4097936|
|       3| 100740|
|       8| 100205|
|       2| 100269|
|       4|  99945|
+--------+-------+



In [67]:
skew_df.rdd.getNumPartitions()

9

In [68]:
skew_df.groupBy(spark_partition_id()).count().show()

+--------------------+------+
|SPARK_PARTITION_ID()| count|
+--------------------+------+
|                   0|602849|
|                   1|603910|
|                   2|620996|
|                   3|621485|
|                   4|600814|
|                   5|601159|
|                   6|602040|
|                   7|746746|
+--------------------+------+



In [69]:
# Ideal Partition Size = 128 MB to 256 MB 

In [70]:
# Adaptive Query Execution 
# Merges the small partition 
# Handle Data Skew  
# Optimize the Join 

In [72]:
spark.conf.set("spark.sql.adaptive.enabled","true")

In [73]:
skew_df.groupBy("RegionId").count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['RegionId], ['RegionId, count(1) AS count#808L]
+- Project [id#530L, CustomerID#537L, CASE WHEN (rand(-7811704332069042389) < 0.8) THEN cast(1 as bigint) ELSE FLOOR((rand(7990485681889895472) * cast(10 as double))) END AS RegionId#750L, Amount#544L]
   +- RepartitionByExpression [CustomerId#537L]
      +- Project [id#530L, CustomerID#537L, RegionId#540L, FLOOR((rand(503387701490677846) * cast(1000 as double))) AS Amount#544L]
         +- Project [id#530L, CustomerID#537L, FLOOR((rand(4123467350201707597) * cast(10 as double))) AS RegionId#540L]
            +- Project [id#530L, FLOOR((rand(89752680378132787) * cast(100000 as double))) AS CustomerID#537L]
               +- Range (1, 5000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
RegionId: bigint, count: bigint
Aggregate [RegionId#750L], [RegionId#750L, count(1) AS count#808L]
+- Project [id#530L, CustomerID#537L, CASE WHEN (rand(-7811704332069042389) < 0.8) THEN cast(1 as bigint) ELSE 